In [1]:
import pandas as pd
import numpy as np
import torch
from transformer_time_series_enc_dec import train_model,InformerForecaster,create_dataloaders,TrainConfig,inverse_transform,_init_weights
import plotly.express as px
import matplotlib.pyplot as plt
import random

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [3]:
import json
from datetime import datetime
import os

def create_loss_plot(train_hist, val_hist, steps_hist, total_params=None):
    """Create a loss plot from training history"""
    df_train = pd.DataFrame({
        "Step": steps_hist,
        "Loss": train_hist,
        "Type": "Train"
    })
    if val_hist:
        val_steps, val_losses = zip(*val_hist)
        df_val = pd.DataFrame({
            "Step": val_steps,
            "Loss": val_losses,
            "Type": "Validation"
        })
        df_loss = pd.concat([df_train, df_val], ignore_index=True)
    else:
        df_loss = df_train
        
    title = "Training and Validation Loss"
    if total_params is not None:
        title += f"\nTotal Parameters: {total_params:,}"
    
    fig = px.line(df_loss, x="Step", y="Loss", color="Type", title=title)
    return fig, df_loss

def create_prediction_plots(model, val_loader, asset_idx, scaler, config, num_batches=10, save_dir=None):
    """Create prediction plots and calculate metrics for validation batches
    
    Args:
        model: The trained model
        val_loader: Validation data loader
        asset_idx: Index of the asset to predict
        scaler: Scaler used for data normalization
        config: Model configuration
        num_batches: Number of batches to visualize
        save_dir: If provided, save plots to this directory
        
    Returns:
        list: List of dictionaries containing metrics for each batch
    """
    metrics = []
    device = next(model.parameters()).device
    
    with torch.no_grad():
        for i, batch in enumerate(val_loader):
            if i >= num_batches:
                break
                
            # Unpack the batch tuple correctly
            x, timestamps = batch
            x = x.to(device)
            timestamps = timestamps.to(device)
            y_pred = model(x, timestamps)
            y_true = x[:, -model.pred_len:, asset_idx]

            # Move to CPU and convert to numpy
            y_pred = y_pred.cpu().numpy()
            y_true = y_true.cpu().numpy()

            # Inverse transform to real prices
            y_pred_price = inverse_transform(y_pred.flatten(), scaler, asset_idx, config["d_input"])
            y_true_price = inverse_transform(y_true.flatten(), scaler, asset_idx, config["d_input"])

            # Create plots
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))

            # Normalised scale
            axes[0].plot(y_true.flatten(), label="Normalised True")
            axes[0].plot(y_pred.flatten(), label="Normalised Predicted")
            axes[0].set_title(f"Batch {i+1} – Normalised")
            axes[0].set_xlabel("Prediction Step")
            axes[0].set_ylabel("Scaled Value")
            axes[0].legend()

            # Real-price scale
            axes[1].plot(y_true_price, label="Real True")
            axes[1].plot(y_pred_price, label="Real Predicted")
            axes[1].set_title(f"Batch {i+1} – Real Prices")
            axes[1].set_xlabel("Prediction Step")
            axes[1].set_ylabel("Price")
            axes[1].legend()

            plt.tight_layout()
            
            # Save or show the plot
            if save_dir:
                plt.savefig(os.path.join(save_dir, f'validation_batch_{i+1}.png'))
                plt.close()
            else:
                plt.show()

            # Calculate metrics
            true = y_true.flatten()
            pred = y_pred.flatten()

            # --- Jaggedness metrics ---
            def mean_abs_diff(x):
                return np.mean(np.abs(np.diff(x)))
            
            pred_jagg = mean_abs_diff(pred)
            true_jagg = mean_abs_diff(true)
            jagg_ratio = pred_jagg / (true_jagg + 1e-8)  # avoid divide by zero
            
            metrics.append({
                'batch': i+1,
                'true_std': float(np.std(true)),
                'pred_std': float(np.std(pred)),
                'mse': float(np.mean((true - pred) ** 2)),
                'mae': float(np.mean(np.abs(true - pred))),
                'true_jaggedness': float(true_jagg),
                'pred_jaggedness': float(pred_jagg),
                'jaggedness_ratio': float(jagg_ratio)
            })
            
    return metrics

def save_experiment_results(config, train_hist, val_hist, steps_hist, model, val_loader_1, asset_idx, scaler, num_batches=10):
    """Save experimental results including losses, plots, and metrics"""
    # Create experiment directory with timestamp
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    exp_dir = f"experiments_{timestamp}"
    os.makedirs(exp_dir, exist_ok=True)
    
    # Calculate and save total learnable parameters
    total_learnable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # Save configuration with model size
    config_with_params = config.copy()
    config_with_params['total_learnable_parameters'] = total_learnable_params
    with open(os.path.join(exp_dir, 'config.json'), 'w') as f:
        json.dump(config_with_params, f, indent=4)
    
    # Create and save loss plot and data
    fig, df_loss = create_loss_plot(train_hist, val_hist, steps_hist, total_learnable_params)
    fig.write_html(os.path.join(exp_dir, 'loss_plot.html'))
    df_loss.to_csv(os.path.join(exp_dir, 'loss_history.csv'))
    
    # Create validation plots and get metrics
    metrics = create_prediction_plots(
        model=model,
        val_loader=val_loader_1,
        asset_idx=asset_idx,
        scaler=scaler,
        config=config,
        num_batches=num_batches,
        save_dir=exp_dir
    )
    
    # Save metrics
    metrics_df = pd.DataFrame(metrics)
    metrics_df.to_csv(os.path.join(exp_dir, 'validation_metrics.csv'), index=False)
    
    # Save model summary information
    with open(os.path.join(exp_dir, 'model_summary.txt'), 'w') as f:
        f.write(f"Total Learnable Parameters: {total_learnable_params:,}\n")
        f.write(f"\nModel Configuration:\n")
        for key, value in config_with_params.items():
            f.write(f"{key}: {value}\n")
    
    return exp_dir

In [4]:
# -----------------------------
# Load and preprocess data
# -----------------------------
csv_path = r"D:\Quan\Quants\Neural Network\financial_attention\1h_data_20220101_20250601.csv"
closes = pd.read_csv(csv_path, index_col=0, parse_dates=True)[['SOL', 'ETH', 'BTC','ADA','XRP','LTC','TRX','LINK','DOT','DOGE']]



In [ ]:
# Define parameter grid
param_grid = {
    'd_model': [16, 32,64],  # Model dimensions
    'distill': [False],   # Whether to use distillation
    'use_time_embedding': [False, True],  # Whether to use time embeddings
    'dropout': [0.05,0.1]  # Dropout rates
}

# Generate all possible combinations
from itertools import product

# Generate all combinations
keys = param_grid.keys()
configs = []
for values in product(*param_grid.values()):
    config_dict = dict(zip(keys, values))
    # Base configuration
    config = {
        "d_input": len(closes.columns),
        "n_heads": 4,
        "enc_layers": 3,
        "dec_layers": 2,
        "enc_len": 96,
        "guiding_len": 48,
        "pred_len": 24,
        "factor": 5,
    }
    # Update with current combination
    config.update(config_dict)
    # Set d_ff to 4x d_model
    config['d_ff'] = config['d_model'] * 4
    configs.append(config)

print(f"Total number of configurations to test: {len(configs)}")
for i, cfg in enumerate(configs):
    print(f"\nConfiguration {i+1}:")
    print(f"d_model: {cfg['d_model']}, d_ff: {cfg['d_ff']}")
    print(f"distill: {cfg['distill']}, use_time_embedding: {cfg['use_time_embedding']}")
    print(f"dropout: {cfg['dropout']}")

Total number of configurations to test: 36

Configuration 1:
d_model: 16, d_ff: 64
distill: False, use_time_embedding: False
dropout: 0.0

Configuration 2:
d_model: 16, d_ff: 64
distill: False, use_time_embedding: False
dropout: 0.05

Configuration 3:
d_model: 16, d_ff: 64
distill: False, use_time_embedding: False
dropout: 0.1

Configuration 4:
d_model: 16, d_ff: 64
distill: False, use_time_embedding: True
dropout: 0.0

Configuration 5:
d_model: 16, d_ff: 64
distill: False, use_time_embedding: True
dropout: 0.05

Configuration 6:
d_model: 16, d_ff: 64
distill: False, use_time_embedding: True
dropout: 0.1

Configuration 7:
d_model: 16, d_ff: 64
distill: True, use_time_embedding: False
dropout: 0.0

Configuration 8:
d_model: 16, d_ff: 64
distill: True, use_time_embedding: False
dropout: 0.05

Configuration 9:
d_model: 16, d_ff: 64
distill: True, use_time_embedding: False
dropout: 0.1

Configuration 10:
d_model: 16, d_ff: 64
distill: True, use_time_embedding: True
dropout: 0.0

Configurat

In [ ]:
def run_experiment(config, exp_dir, exp_name):
    """Run a single experiment with given configuration"""
    # Set seeds for reproducibility
    set_seed(42)
    
    # Create data loaders
    train_loader, val_loader, scaler, asset_idx = create_dataloaders(
        closes, 
        enc_len=config["enc_len"],
        pred_len=config["pred_len"],
        batch_size=32,
        val_batch_size=32, 
        val_ratio=0.1, 
        asset_name="SOL"
    )
    
    # Initialize model
    model = InformerForecaster(config, asset_index=asset_idx)
    model.apply(_init_weights)
    learnable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # Training configuration
    tcfg = TrainConfig(
        learning_rate=1e-4,
        weight_decay=0.01,
        max_steps=10000,
        warmup_steps=200,
        use_amp=True,
        device="cuda",
        patience=15,
        min_delta=0.0001
    )
    
    # Train model
    model, train_hist, val_hist, steps_hist = train_model(
        model, train_loader, val_loader, tcfg, asset_index=asset_idx
    )
    
    # Get early stopping step
    early_stop_step = len(steps_hist)*10
    
    # Create validation loader for visualization
    _, val_loader_1, _, _ = create_dataloaders(
        closes, 
        enc_len=config["enc_len"],
        pred_len=config["pred_len"],
        batch_size=32, 
        val_batch_size=1,
        val_shuffle=True,
        val_ratio=0.1, 
        asset_name="SOL"
    )
    
    # Create experiment-specific directory
    exp_subdir = os.path.join(exp_dir, exp_name)
    os.makedirs(exp_subdir, exist_ok=True)
    
    # Save model and configuration
    model_save_path = os.path.join(exp_subdir, 'model.pth')
    torch.save({
        'model_state_dict': model.state_dict(),
        'config': config,
        'total_params': learnable_params,
        'train_hist': train_hist,
        'val_hist': val_hist,
        'steps_hist': steps_hist
    }, model_save_path)
    
    # Create and save plots
    fig, df_loss = create_loss_plot(
        train_hist, 
        val_hist, 
        steps_hist,
        total_params=learnable_params
    )
    fig.write_html(os.path.join(exp_subdir, 'loss_plot.html'))
    df_loss.to_csv(os.path.join(exp_subdir, 'loss_history.csv'))
    
    # Calculate and save metrics
    metrics = create_prediction_plots(
        model=model,
        val_loader=val_loader_1,
        asset_idx=asset_idx,
        scaler=scaler,
        config=config,
        num_batches=10,
        save_dir=exp_subdir
    )
    
    metrics_df = pd.DataFrame(metrics)
    metrics_df.to_csv(os.path.join(exp_subdir, 'metrics.csv'), index=False)
    
    # Save summary metrics
    summary_metrics = metrics_df.mean().round(4)
    
    # Get final losses
    final_train_loss = train_hist[-1]
    final_val_loss = val_hist[-1][1] if val_hist else float('nan')
    
    return {
        'Experiment no': exp_name,
        'd_model': config['d_model'],
        'd_ff': config['d_ff'],
        'distill': config['distill'],
        'time embedding': config['use_time_embedding'],
        'dropout': config['dropout'],
        'learnable params': learnable_params,
        'Early Stopping Step': early_stop_step,
        'train loss': final_train_loss,
        'validation loss': final_val_loss,
        'jaggedness_ratio (pred/real)': float(summary_metrics['jaggedness_ratio']),
        'MSE': float(summary_metrics['mse']),
        'MAE': float(summary_metrics['mae'])
    }

In [ ]:
# Create main experiments directory with timestamp
exp_dir = f"experiments_grid_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(exp_dir, exist_ok=True)

# Run all experiments
results = []
for i, config in enumerate(configs):
    print(f"\nRunning experiment {i+1}/{len(configs)}")
    print("Configuration:", config)
    
    # Create experiment name from parameters
    exp_name = (f"d{config['d_model']}_"
               f"{'dist' if config['distill'] else 'nodist'}_"
               f"{'time' if config['use_time_embedding'] else 'notime'}_"
               f"drop{config['dropout']}")
    
    # Run experiment
    try:
        result = run_experiment(config, exp_dir, exp_name)
        results.append(result)
        print(f"Experiment {exp_name} completed successfully")
        print(f"Final val loss: {result['validation loss']:.6f}")
        print(f"MSE: {result['MSE']:.6f}")
    except Exception as e:
        print(f"Experiment {exp_name} failed with error: {str(e)}")
        continue

# Create results summary with specific column order
columns = [
    'Experiment no', 
    # Model Params
    'd_model', 'd_ff', 'distill', 'time embedding', 'dropout', 'learnable params',
    # Results
    'Early Stopping Step', 'train loss', 'validation loss', 
    'jaggedness_ratio (pred/real)', 'MSE', 'MAE'
]

results_df = pd.DataFrame(results)[columns]

# Save results with proper formatting
results_df.to_csv(os.path.join(exp_dir, 'all_results.csv'), index=False, float_format='%.6f')

# Create summary visualizations
fig = px.scatter(results_df, 
                 x='validation loss', 
                 y='MSE',
                 hover_data=columns,
                 title='Validation Loss vs MSE across experiments',
                 labels={'Experiment no': 'Experiment Name'})  # Update label
fig.write_html(os.path.join(exp_dir, 'results_scatter.html'))

# Save experiment configuration summary
config_summary = {
    'timestamp': datetime.now().strftime('%Y%m%d_%H%M%S'),
    'total_experiments': len(configs),
    'parameter_grid': param_grid,
    'base_config': {k: v for k, v in configs[0].items() if k not in param_grid},
    'experiment_names': [r['Experiment no'] for r in results]
}
with open(os.path.join(exp_dir, 'experiment_config.json'), 'w') as f:
    json.dump(config_summary, f, indent=4)

# Print best models by different metrics
print("\nBest models by validation loss:")
print(results_df.nsmallest(3, 'validation loss')[columns])

print("\nBest models by MSE:")
print(results_df.nsmallest(3, 'MSE')[columns])


Running experiment 1/36
Configuration: {'d_input': 10, 'n_heads': 4, 'enc_layers': 3, 'dec_layers': 2, 'enc_len': 96, 'guiding_len': 48, 'pred_len': 24, 'factor': 5, 'd_model': 16, 'distill': False, 'use_time_embedding': False, 'dropout': 0.0, 'd_ff': 64}


2025-10-21 09:46:06,605 | INFO | Using fused AdamW: True
2025-10-21 09:46:08,611 | INFO | step 0 | train loss 0.906979 | lr 0.000e+00 | tok/s 3840000000000.0
2025-10-21 09:46:09,724 | INFO | step 0 | VALIDATION loss 1.149278 | best 1.149278 | patience 0/15
2025-10-21 09:46:10,037 | INFO | step 10 | train loss 0.822006 | lr 5.000e-06 | tok/s 2692.8
2025-10-21 09:46:10,347 | INFO | step 20 | train loss 0.914095 | lr 1.000e-05 | tok/s 12434.4
2025-10-21 09:46:10,656 | INFO | step 30 | train loss 0.849398 | lr 1.500e-05 | tok/s 12444.0
2025-10-21 09:46:10,959 | INFO | step 40 | train loss 1.048782 | lr 2.000e-05 | tok/s 12673.3
2025-10-21 09:46:11,252 | INFO | step 50 | train loss 0.802548 | lr 2.500e-05 | tok/s 13105.8
2025-10-21 09:46:11,535 | INFO | step 60 | train loss 1.144691 | lr 3.000e-05 | tok/s 13617.0
2025-10-21 09:46:11,833 | INFO | step 70 | train loss 1.208525 | lr 3.500e-05 | tok/s 12885.9
2025-10-21 09:46:12,152 | INFO | step 80 | train loss 0.874169 | lr 4.000e-05 | tok/s 

Experiment d16_nodist_notime_drop0.0 completed successfully
Final val loss: 0.035407
MSE: 0.033400

Running experiment 2/36
Configuration: {'d_input': 10, 'n_heads': 4, 'enc_layers': 3, 'dec_layers': 2, 'enc_len': 96, 'guiding_len': 48, 'pred_len': 24, 'factor': 5, 'd_model': 16, 'distill': False, 'use_time_embedding': False, 'dropout': 0.05, 'd_ff': 64}


2025-10-21 09:55:43,680 | INFO | step 0 | VALIDATION loss 1.149279 | best 1.149279 | patience 0/15
2025-10-21 09:55:43,971 | INFO | step 10 | train loss 0.821998 | lr 5.000e-06 | tok/s 2864.5
2025-10-21 09:55:44,251 | INFO | step 20 | train loss 0.914122 | lr 1.000e-05 | tok/s 13763.5
2025-10-21 09:55:44,518 | INFO | step 30 | train loss 0.849411 | lr 1.500e-05 | tok/s 14408.4
2025-10-21 09:55:44,806 | INFO | step 40 | train loss 1.048728 | lr 2.000e-05 | tok/s 13376.5
2025-10-21 09:55:45,079 | INFO | step 50 | train loss 0.802572 | lr 2.500e-05 | tok/s 14117.6
2025-10-21 09:55:45,347 | INFO | step 60 | train loss 1.144640 | lr 3.000e-05 | tok/s 14382.1
2025-10-21 09:55:45,638 | INFO | step 70 | train loss 1.208517 | lr 3.500e-05 | tok/s 13195.8
2025-10-21 09:55:45,926 | INFO | step 80 | train loss 0.874186 | lr 4.000e-05 | tok/s 13379.9
2025-10-21 09:55:46,195 | INFO | step 90 | train loss 0.720613 | lr 4.500e-05 | tok/s 14328.4
2025-10-21 09:55:46,454 | INFO | step 100 | train loss 1

Experiment d16_nodist_notime_drop0.05 completed successfully
Final val loss: 0.040036
MSE: 0.038200

Running experiment 3/36
Configuration: {'d_input': 10, 'n_heads': 4, 'enc_layers': 3, 'dec_layers': 2, 'enc_len': 96, 'guiding_len': 48, 'pred_len': 24, 'factor': 5, 'd_model': 16, 'distill': False, 'use_time_embedding': False, 'dropout': 0.1, 'd_ff': 64}


...

2025-10-21 13:26:02,411 | INFO | step 0 | VALIDATION loss 1.229867 | best 1.229867 | patience 0/15
2025-10-21 13:26:03,187 | INFO | step 10 | train loss 0.925105 | lr 5.000e-06 | tok/s 1009.4
2025-10-21 13:26:03,883 | INFO | step 20 | train loss 0.967093 | lr 1.000e-05 | tok/s 5520.3
2025-10-21 13:26:04,553 | INFO | step 30 | train loss 0.983395 | lr 1.500e-05 | tok/s 5751.0
2025-10-21 13:26:05,213 | INFO | step 40 | train loss 0.739574 | lr 2.000e-05 | tok/s 5824.6
2025-10-21 13:26:05,887 | INFO | step 50 | train loss 1.115147 | lr 2.500e-05 | tok/s 5708.1
2025-10-21 13:26:06,549 | INFO | step 60 | train loss 1.245282 | lr 3.000e-05 | tok/s 5811.4
2025-10-21 13:26:07,224 | INFO | step 70 | train loss 0.824493 | lr 3.500e-05 | tok/s 5703.3
2025-10-21 13:26:07,894 | INFO | step 80 | train loss 0.834508 | lr 4.000e-05 | tok/s 5745.2
2025-10-21 13:26:08,546 | INFO | step 90 | train loss 0.893675 | lr 4.500e-05 | tok/s 5898.1
2025-10-21 13:26:09,229 | INFO | step 100 | train loss 0.887043 

Experiment d64_dist_time_drop0.05 completed successfully
Final val loss: 0.070882
MSE: 0.013600

Running experiment 36/36
Configuration: {'d_input': 10, 'n_heads': 4, 'enc_layers': 3, 'dec_layers': 2, 'enc_len': 96, 'guiding_len': 48, 'pred_len': 24, 'factor': 5, 'd_model': 64, 'distill': True, 'use_time_embedding': True, 'dropout': 0.1, 'd_ff': 256}


2025-10-21 13:28:59,955 | INFO | step 0 | VALIDATION loss 1.229867 | best 1.229867 | patience 0/15
2025-10-21 13:29:00,327 | INFO | step 10 | train loss 0.924992 | lr 5.000e-06 | tok/s 2173.0
2025-10-21 13:29:00,725 | INFO | step 20 | train loss 0.967087 | lr 1.000e-05 | tok/s 9661.2
2025-10-21 13:29:01,113 | INFO | step 30 | train loss 0.983449 | lr 1.500e-05 | tok/s 9976.2
2025-10-21 13:29:01,480 | INFO | step 40 | train loss 0.739640 | lr 2.000e-05 | tok/s 10494.0
2025-10-21 13:29:01,863 | INFO | step 50 | train loss 1.115166 | lr 2.500e-05 | tok/s 10043.2
2025-10-21 13:29:02,227 | INFO | step 60 | train loss 1.245244 | lr 3.000e-05 | tok/s 10551.2
2025-10-21 13:29:02,600 | INFO | step 70 | train loss 0.824360 | lr 3.500e-05 | tok/s 10295.7
2025-10-21 13:29:03,060 | INFO | step 80 | train loss 0.834706 | lr 4.000e-05 | tok/s 8344.4
2025-10-21 13:29:03,541 | INFO | step 90 | train loss 0.893964 | lr 4.500e-05 | tok/s 8006.6
2025-10-21 13:29:03,934 | INFO | step 100 | train loss 0.887

Experiment d64_dist_time_drop0.1 completed successfully
Final val loss: 0.072013
MSE: 0.014400

Best models by validation loss:
                 Experiment no  d_model  d_ff  distill  time embedding  \
26   d64_nodist_notime_drop0.1       64   256    False           False   
24   d64_nodist_notime_drop0.0       64   256    False           False   
25  d64_nodist_notime_drop0.05       64   256    False           False   

    dropout  learnable params  Early Stopping Step  train loss  \
26     0.10            284929                  561    0.002226   
24     0.00            284929                  561    0.001702   
25     0.05            284929                  661    0.001245   

    validation loss  jaggedness_ratio (pred/real)     MSE     MAE  
26         0.003846                        0.8003  0.0014  0.0306  
24         0.004951                        0.7946  0.0013  0.0281  
25         0.006008                        0.8314  0.0034  0.0376  

Best models by MSE:
                 

: 